# K-Means Clustering: Finding Groups in Data

Clustering is a foundational technique in unsupervised machine learning. Unlike supervised learning where we have labeled target variables, clustering algorithms discover hidden patterns and groupings in data without any prior labels. This notebook explores K-Means — the most widely used clustering algorithm — from intuition to implementation, evaluation, and real-world application.

## 1. What is Clustering? Supervised vs Unsupervised Learning

**Supervised Learning:** The model trains on labeled data where each example has an input $X$ and a known output $y$. The goal is to learn a mapping $f: X \to y$. Examples: regression, classification.

**Unsupervised Learning:** The model trains on data with no labels. The algorithm must find structure, patterns, or groupings on its own. Examples: clustering, dimensionality reduction, anomaly detection.

**Clustering** partitions a dataset into groups (clusters) such that:
- **Intra-cluster similarity** is high (points within the same cluster are close together).
- **Inter-cluster similarity** is low (points in different clusters are far apart).

K-Means is a centroid-based clustering algorithm. It assumes clusters are spherical and roughly equal in size, and it works best when clusters are well-separated.

## 2. Intuition: How K-Means Works

The core idea is simple:
1. Choose $K$ initial centroids (cluster centers).
2. **Assignment step:** Assign each data point to its nearest centroid.
3. **Update step:** Recompute each centroid as the mean of all points assigned to it.
4. Repeat steps 2-3 until convergence (centroids stop moving significantly).

The algorithm minimizes the **within-cluster sum of squares (WCSS)**, also called **inertia**:

$$
\text{Inertia} = \sum_{i=1}^{K} \sum_{x \in C_i} \| x - \mu_i \|^2
$$

where $C_i$ is the $i$-th cluster and $\mu_i$ is its centroid.

## 3. The K-Means Algorithm Step-by-Step

Let's walk through a simple 2D example with $K=2$ and 6 data points:
- Points: A(1,2), B(1,4), C(3,2), D(7,6), E(8,5), F(8,7)

**Step 0 — Choose K:** We set $K=2$.

**Step 1 — Initialize centroids:** Randomly pick two points as initial centroids. Suppose $\mu_1 = A(1,2)$ and $\mu_2 = D(7,6)$.

**Step 2 — Assign clusters:** Compute Euclidean distance from each point to both centroids.
- A(1,2): dist to $\mu_1$ = 0, dist to $\mu_2$ = 7.21 → Cluster 1
- B(1,4): dist to $\mu_1$ = 2.0, dist to $\mu_2$ = 6.32 → Cluster 1
- C(3,2): dist to $\mu_1$ = 2.0, dist to $\mu_2$ = 5.66 → Cluster 1
- D(7,6): dist to $\mu_1$ = 7.21, dist to $\mu_2$ = 0 → Cluster 2
- E(8,5): dist to $\mu_1$ = 7.62, dist to $\mu_2$ = 1.41 → Cluster 2
- F(8,7): dist to $\mu_1$ = 8.60, dist to $\mu_2$ = 1.41 → Cluster 2

**Step 3 — Update centroids:**
- $\mu_1$ = mean of (1,2), (1,4), (3,2) = (1.67, 2.67)
- $\mu_2$ = mean of (7,6), (8,5), (8,7) = (7.67, 6.00)

**Step 4 — Repeat:** Reassign points to the new centroids. Points will stay in the same clusters. Centroids stabilize → **convergence**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs, make_classification
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score, calinski_harabasz_score
from sklearn_extra.cluster import KMedoids
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('All imports successful.')

## 4. Implementing K-Means from Scratch

In [ ]:
class KMeansScratch:
    """K-Means clustering implemented from scratch."""
    
    def __init__(self, n_clusters=3, max_iters=100, tol=1e-4, random_seed=None):
        self.n_clusters = n_clusters
        self.max_iters = max_iters
        self.tol = tol
        self.random_seed = random_seed
        self.centroids = None
        self.labels_ = None
        self.inertia_ = None
        self.n_iter_ = 0
    
    def _euclidean(self, a, b):
        return np.sqrt(np.sum((a - b) ** 2))
    
    def _initialize_centroids(self, X):
        rng = np.random.RandomState(self.random_seed)
        indices = rng.choice(len(X), self.n_clusters, replace=False)
        return X[indices]
    
    def _assign_clusters(self, X):
        distances = np.zeros((len(X), self.n_clusters))
        for i, point in enumerate(X):
            for j, centroid in enumerate(self.centroids):
                distances[i, j] = self._euclidean(point, centroid)
        return np.argmin(distances, axis=1)
    
    def _update_centroids(self, X, labels):
        new_centroids = np.zeros_like(self.centroids)
        for j in range(self.n_clusters):
            mask = labels == j
            if np.sum(mask) == 0:
                new_centroids[j] = self.centroids[j]
            else:
                new_centroids[j] = X[mask].mean(axis=0)
        return new_centroids
    
    def _compute_inertia(self, X, labels):
        inertia = 0.0
        for j in range(self.n_clusters):
            mask = labels == j
            if np.sum(mask) > 0:
                inertia += np.sum((X[mask] - self.centroids[j]) ** 2)
        return inertia
    
    def fit(self, X):
        self.centroids = self._initialize_centroids(X)
        
        for iteration in range(self.max_iters):
            self.n_iter_ = iteration + 1
            labels = self._assign_clusters(X)
            new_centroids = self._update_centroids(X, labels)
            
            shift = np.sum([self._euclidean(self.centroids[j], new_centroids[j])
                           for j in range(self.n_clusters)])
            
            self.centroids = new_centroids
            
            if shift < self.tol:
                break
        
        self.labels_ = self._assign_clusters(X)
        self.inertia_ = self._compute_inertia(X, self.labels_)
        return self
    
    def predict(self, X):
        return self._assign_clusters(X)


# Demo on synthetic data
X_demo, y_demo = make_blobs(n_samples=150, centers=3, cluster_std=1.5, random_state=42)

scratch_model = KMeansScratch(n_clusters=3, random_seed=42)
scratch_model.fit(X_demo)

print(f'From-scratch K-Means converged in {scratch_model.n_iter_} iterations')
print(f'Inertia: {scratch_model.inertia_:.2f}')
print(f'Cluster sizes: {np.bincount(scratch_model.labels_)}')

In [ ]:
# Compare with sklearn
sk_model = KMeans(n_clusters=3, n_init=10, random_state=42)
sk_model.fit(X_demo)

print(f'sklearn inertia: {sk_model.inertia_:.2f}')
print(f'sklearn cluster sizes: {np.bincount(sk_model.labels_)}')
print(f'Centroids match (approx): {np.allclose(np.sort(scratch_model.centroids.sum(axis=1)), 
                                               np.sort(sk_model.cluster_centers_.sum(axis=1)), atol=0.1)}')

## 5. Using scikit-learn's KMeans

In [ ]:
X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=1.2, random_state=42)

kmeans = KMeans(n_clusters=4, init='k-means++', n_init=10, max_iter=300, random_state=42)
kmeans.fit(X)

print(f'Labels shape: {kmeans.labels_.shape}')
print(f'Cluster centers:\n{kmeans.cluster_centers_}')
print(f'Inertia: {kmeans.inertia_:.2f}')
print(f'Number of iterations: {kmeans.n_iter_}')

## 6. The Elbow Method for Choosing K

The elbow method runs K-Means for a range of $K$ values and plots the inertia (WCSS). The optimal $K$ is at the "elbow" — where the rate of decrease sharply changes.

In [ ]:
inertias = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='K=4 (true clusters)')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (WCSS)')
plt.title('Elbow Method for Optimal K')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. Silhouette Score and Silhouette Plot

The **silhouette score** measures how similar a point is to its own cluster compared to other clusters. It ranges from -1 to 1:
- **+1:** Point is far from neighboring clusters (well-clustered).
- **0:** Point is on the decision boundary.
- **-1:** Point may be assigned to the wrong cluster.

For each point $i$:
- $a(i)$ = mean distance to points in the same cluster.
- $b(i)$ = mean distance to points in the nearest neighboring cluster.
- $s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$

The overall silhouette score is the mean of $s(i)$ across all points.

In [ ]:
def plot_silhouette(X, labels, n_clusters):
    """Plot silhouette analysis."""
    silhouette_vals = silhouette_samples(X, labels)
    avg_score = silhouette_score(X, labels)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    
    y_lower = 10
    for i in range(n_clusters):
        cluster_vals = silhouette_vals[labels == i]
        cluster_vals.sort()
        cluster_size = len(cluster_vals)
        y_upper = y_lower + cluster_size
        
        ax1.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_vals,
                          alpha=0.7, label=f'Cluster {i}')
        ax1.axvline(x=avg_score, color='red', linestyle='--', 
                    label=f'Avg: {avg_score:.3f}')
        y_lower = y_upper + 10
    
    ax1.set_xlabel('Silhouette Coefficient')
    ax1.set_ylabel('Cluster')
    ax1.set_title('Silhouette Plot')
    ax1.legend(loc='best')
    ax1.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
    
    for i in range(n_clusters):
        ax2.scatter(X[labels == i, 0], X[labels == i, 1], 
                   label=f'Cluster {i}', alpha=0.6, edgecolors='k', s=50)
    
    ax2.set_title(f'Clusters (Silhouette Score: {avg_score:.3f})')
    ax2.set_xlabel('Feature 1')
    ax2.set_ylabel('Feature 2')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
    
    return avg_score


score = plot_silhouette(X, kmeans.labels_, 4)
print(f'Silhouette Score for K=4: {score:.4f}')

In [ ]:
# Compare silhouette scores across K
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X)
    sil = silhouette_score(X, labels)
    silhouette_scores.append(sil)
    print(f'K={k}: Silhouette = {sil:.4f}')

plt.figure(figsize=(10, 5))
plt.plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
plt.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='K=4 (true)')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs Number of Clusters')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Davies-Bouldin Index and Calinski-Harabasz Index

**Davies-Bouldin Index (DBI):** The average similarity between each cluster and its most similar one. Lower values indicate better separation (minimum is 0).

$$
DB = \frac{1}{K} \sum_{i=1}^{K} \max_{j \neq i} \left( \frac{\sigma_i + \sigma_j}{d(\mu_i, \mu_j)} \right)
$$

where $\sigma_i$ is the average distance from points in cluster $i$ to its centroid $\mu_i$.

**Calinski-Harabasz Index (CHI):** The ratio of between-cluster dispersion to within-cluster dispersion. Higher values indicate better-defined clusters.

$$
CH = \frac{\text{tr}(B_K)}{\text{tr}(W_K)} \times \frac{N - K}{K - 1}
$$

where $B_K$ is the between-cluster covariance matrix and $W_K$ is the within-cluster covariance matrix.

In [ ]:
print(f'Davies-Bouldin Index (K=4): {davies_bouldin_score(X, kmeans.labels_):.4f}')
print(f'Calinski-Harabasz Index (K=4): {calinski_harabasz_score(X, kmeans.labels_):.2f}')

# Compare across K
results = []
for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X)
    dbi = davies_bouldin_score(X, labels)
    chi = calinski_harabasz_score(X, labels)
    results.append({'K': k, 'DBI': dbi, 'CHI': chi})

df_metrics = pd.DataFrame(results)
print('\nMetrics across K:')
print(df_metrics.to_string(index=False))

## 9. K-Means++ Initialization

Standard K-Means initializes centroids randomly, which can lead to poor clustering or slow convergence. **K-Means++** improves initialization:

1. Pick the first centroid uniformly at random from the data points.
2. For each remaining point, compute $D(x)$, the distance to its nearest already-chosen centroid.
3. Choose the next centroid with probability proportional to $D(x)^2$ (points farther away are more likely to be selected).
4. Repeat until all $K$ centroids are chosen.

This spreads out initial centroids and leads to better, more consistent results. It is the default `init='k-means++'` in scikit-learn.

In [ ]:
# Compare random vs k-means++ initialization
np.random.seed(42)
X_vis, _ = make_blobs(n_samples=200, centers=4, cluster_std=2.0, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (init_method, title) in enumerate([('random', 'Random Initialization'), 
                                            ('k-means++', 'K-Means++ Initialization')]):
    inertias_run = []
    for run in range(30):
        km = KMeans(n_clusters=4, init=init_method, n_init=1, max_iter=300, random_state=run)
        km.fit(X_vis)
        inertias_run.append(km.inertia_)
    
    axes[idx].hist(inertias_run, bins=12, edgecolor='black', alpha=0.7)
    axes[idx].axvline(x=np.min(inertias_run), color='green', linestyle='--', 
                     label=f'Best: {np.min(inertias_run):.0f}')
    axes[idx].set_xlabel('Inertia')
    axes[idx].set_ylabel('Frequency')
    axes[idx].set_title(title)
    axes[idx].legend()

plt.tight_layout()
plt.show()

print(f'Random init  - Mean inertia: {np.mean(inertias_run):.0f}, Std: {np.std(inertias_run):.0f}')
print(f'K-Means++ init avoids poor local minima and reduces variance across runs.')

## 10. Limitations of K-Means

1. **Sensitivity to initialization:** Poor initialization can lead to suboptimal clustering. Mitigated by K-Means++ and multiple restarts (`n_init`).
2. **Assumes spherical clusters:** K-Means works poorly with elongated, irregular, or non-convex cluster shapes.
3. **Equal cluster sizes bias:** The algorithm implicitly assumes clusters have roughly equal variance.
4. **Scale dependency:** Features with larger scales dominate distance calculations.
5. **Sensitive to outliers:** Outliers pull centroids toward themselves.
6. **Requires specifying K:** The number of clusters must be chosen beforehand.
7. **Hard assignment:** Each point belongs to exactly one cluster (no membership probability).

## 11. Feature Scaling Importance

Since K-Means relies on Euclidean distance, features with larger magnitudes will disproportionately influence the clustering. Always scale features before applying K-Means.

In [ ]:
# Create data with different scales
np.random.seed(42)
n_samples = 200
age = np.random.randint(18, 70, size=n_samples)
income = np.random.randint(20000, 150000, size=n_samples)  # much larger scale
spending = np.random.randint(1, 100, size=n_samples)

X_unscaled = np.column_stack([age, income, spending])

# K-Means on unscaled data
km_unscaled = KMeans(n_clusters=3, random_state=42)
labels_unscaled = km_unscaled.fit_predict(X_unscaled)

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_unscaled)

# K-Means on scaled data
km_scaled = KMeans(n_clusters=3, random_state=42)
labels_scaled = km_scaled.fit_predict(X_scaled)

print('Feature ranges (before scaling):')
print(f'  Age: {age.min()}-{age.max()}')
print(f'  Income: {income.min()}-{income.max()}')
print(f'  Spending: {spending.min()}-{spending.max()}')
print(f'\nCluster centers (unscaled):\n{km_unscaled.cluster_centers_}')
print(f'\nCluster centers (scaled, back-transformed):\n{scaler.inverse_transform(km_scaled.cluster_centers_)}')

## 12. Visualizing Clusters in 2D

When data has more than 2 dimensions, we can use the first two principal components or pick pairs of features for visualization.

In [ ]:
def plot_clusters_2d(X, labels, centroids=None, title='Cluster Visualization'):
    plt.figure(figsize=(10, 7))
    scatter = plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', 
                         edgecolors='k', s=60, alpha=0.7)
    if centroids is not None:
        plt.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='X', 
                   s=200, linewidths=2, edgecolors='black', label='Centroids')
    plt.colorbar(scatter, label='Cluster')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


# Before vs After scaling visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(X_unscaled[:, 0], X_unscaled[:, 1], c=labels_unscaled, 
                cmap='viridis', edgecolors='k', s=50, alpha=0.7)
axes[0].scatter(km_unscaled.cluster_centers_[:, 0], km_unscaled.cluster_centers_[:, 1],
                c='red', marker='X', s=200, edgecolors='black')
axes[0].set_title('Unscaled: Age vs Income')
axes[0].set_xlabel('Age'); axes[0].set_ylabel('Income')

axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels_scaled, 
                cmap='viridis', edgecolors='k', s=50, alpha=0.7)
axes[1].scatter(km_scaled.cluster_centers_[:, 0], km_scaled.cluster_centers_[:, 1],
                c='red', marker='X', s=200, edgecolors='black')
axes[1].set_title('Scaled: Age vs Income (Standardized)')
axes[1].set_xlabel('Age (scaled)'); axes[1].set_ylabel('Income (scaled)')

plt.tight_layout()
plt.show()

## 13. Real-World Example: Customer Segmentation

We generate synthetic customer data with age, annual income, and spending score (1-100). The goal is to identify distinct customer segments for targeted marketing.

In [ ]:
np.random.seed(42)
n_customers = 500

# Segment A: Young, low income, moderate spending
seg_a = np.column_stack([
    np.random.normal(25, 5, 150),
    np.random.normal(30000, 8000, 150),
    np.random.normal(60, 15, 150)
])

# Segment B: Middle-aged, high income, high spending
seg_b = np.column_stack([
    np.random.normal(45, 6, 175),
    np.random.normal(85000, 15000, 175),
    np.random.normal(85, 10, 175)
])

# Segment C: Senior, moderate income, low spending
seg_c = np.column_stack([
    np.random.normal(62, 5, 175),
    np.random.normal(50000, 12000, 175),
    np.random.normal(25, 12, 175)
])

X_customers = np.vstack([seg_a, seg_b, seg_c])
y_true_customers = np.array([0]*150 + [1]*175 + [2]*175)
np.random.shuffle(np.random.permutation(len(X_customers)))

customer_df = pd.DataFrame(X_customers, columns=['Age', 'Annual_Income', 'Spending_Score'])
customer_df['True_Segment'] = y_true_customers
print(customer_df.head(10))
print(f'\nDataset shape: {customer_df.shape}')

In [ ]:
# Scale features
scaler_cust = StandardScaler()
X_cust_scaled = scaler_cust.fit_transform(customer_df[['Age', 'Annual_Income', 'Spending_Score']])

# Find optimal K
inertias_cust = []
sil_scores_cust = []
K_test = range(2, 10)

for k in K_test:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_cust_scaled)
    inertias_cust.append(km.inertia_)
    sil_scores_cust.append(silhouette_score(X_cust_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_test, inertias_cust, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method - Customer Data')
axes[0].grid(True, alpha=0.3)

axes[1].plot(K_test, sil_scores_cust, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score - Customer Data')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Silhouette scores:', {k: f'{s:.4f}' for k, s in zip(K_test, sil_scores_cust)})

In [ ]:
# Apply K-Means with K=3
final_kmeans = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)
customer_labels = final_kmeans.fit_predict(X_cust_scaled)

customer_df['Cluster'] = customer_labels

# Profile clusters
cluster_profile = customer_df.groupby('Cluster').agg({
    'Age': ['mean', 'std'],
    'Annual_Income': ['mean', 'std'],
    'Spending_Score': ['mean', 'std'],
    'True_Segment': 'count'
}).round(1)

print('Cluster Profiles:')
print(cluster_profile)
print(f'\nAdjusted Rand Index (vs true segments): '
      f'{adjusted_rand_score(y_true_customers, customer_labels):.3f}')

In [ ]:
# Visualize customer segments in 2D (using first two features: Age, Income)
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_cust_scaled)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# True segments
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_true_customers, 
                          cmap='Set2', edgecolors='k', s=50, alpha=0.7)
axes[0].set_title('True Customer Segments')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
plt.colorbar(scatter1, ax=axes[0])

# K-Means clusters
scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=customer_labels, 
                          cmap='Set2', edgecolors='k', s=50, alpha=0.7)
axes[1].scatter(pca.transform(final_kmeans.cluster_centers_)[:, 0],
                pca.transform(final_kmeans.cluster_centers_)[:, 1],
                c='red', marker='X', s=200, linewidths=2, edgecolors='black', label='Centroids')
axes[1].set_title('K-Means Clusters (PCA-reduced)')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
plt.colorbar(scatter2, ax=axes[1])
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Explained variance ratio: {pca.explained_variance_ratio_}')

## 14. Comparing K-Means with MiniBatchKMeans

**MiniBatchKMeans** processes small random subsets (mini-batches) of data at each iteration, making it significantly faster on large datasets. It usually converges to slightly higher inertia but with a major speedup.

In [ ]:
import time

# Large dataset
X_large, _ = make_blobs(n_samples=50000, centers=5, cluster_std=1.5, random_state=42)

# Standard K-Means
start = time.time()
km_full = KMeans(n_clusters=5, init='k-means++', n_init=3, random_state=42)
km_full.fit(X_large)
t_full = time.time() - start

# MiniBatchKMeans
start = time.time()
mbkm = MiniBatchKMeans(n_clusters=5, init='k-means++', n_init=3, 
                       batch_size=1024, random_state=42)
mbkm.fit(X_large)
t_mb = time.time() - start

print(f'Standard K-Means:     {t_full:.2f}s, Inertia: {km_full.inertia_:.0f}')
print(f'MiniBatchKMeans:      {t_mb:.2f}s, Inertia: {mbkm.inertia_:.0f}')
print(f'Speedup: {t_full / t_mb:.1f}x')

## 15. K-Medoids and Alternative Algorithms

**K-Medoids (PAM — Partitioning Around Medoids):**
- Instead of centroids (means), K-Medoids uses actual data points as cluster centers (medoids).
- More robust to outliers than K-Means since medoids are not pulled by extreme values.
- Works with arbitrary distance metrics (not just Euclidean).
- More computationally expensive than K-Means.

**When to use alternatives:**

| Algorithm | Best for |
|-----------|----------|
| **K-Means** | Spherical clusters, large datasets, numerical features |
| **K-Medoids** | Robustness to outliers, categorical data (Gower distance) |
| **DBSCAN** | Arbitrary shapes, noise detection, unknown K |
| **Hierarchical** | Hierarchical structure, dendrogram visualization |
| **Gaussian Mixture** | Soft assignment, elliptical clusters |
| **Spectral Clustering** | Non-convex clusters, graph-structured data |

In [ ]:
# Quick comparison: K-Means vs K-Medoids (requires sklearn-extra)
try:
    # K-Medoids
    kmedoids = KMedoids(n_clusters=4, metric='euclidean', random_state=42)
    medoid_labels = kmedoids.fit_predict(X)
    medoid_score = silhouette_score(X, medoid_labels)
    
    # K-Means
    km_small = KMeans(n_clusters=4, random_state=42)
    km_labels = km_small.fit_predict(X)
    km_score = silhouette_score(X, km_labels)
    
    print(f'K-Means  Silhouette: {km_score:.4f} | Medoids: {kmedoids.cluster_centers_}')
    print(f'K-Medoids Silhouette: {medoid_score:.4f} | Medoids: {kmedoids.cluster_centers_}')
except Exception as e:
    print(f'K-Medoids comparison skipped (install sklearn-extra if desired): {e}')

## 16. Summary: Cluster Centers Visualization

A summary visualization showing the final cluster centers for our customer segmentation.

In [ ]:
# Radar chart-like bar plot of cluster centers
centers_df = pd.DataFrame(
    scaler_cust.inverse_transform(final_kmeans.cluster_centers_),
    columns=['Age', 'Annual_Income', 'Spending_Score']
)
centers_df['Cluster'] = ['Segment 0', 'Segment 1', 'Segment 2']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
features = ['Age', 'Annual_Income', 'Spending_Score']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

for idx, feature in enumerate(features):
    bars = axes[idx].bar(centers_df['Cluster'], centers_df[feature], 
                        color=colors, edgecolor='black', alpha=0.8)
    axes[idx].set_title(f'Cluster Centers: {feature}', fontsize=13, fontweight='bold')
    axes[idx].set_ylabel(feature)
    
    for bar, val in zip(bars, centers_df[feature]):
        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(centers_df[feature])*0.02,
                      f'{val:.0f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.suptitle('Customer Segmentation: Cluster Center Profiles', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print('Segment 0: Young adults, low income, moderate spending')
print('Segment 1: Middle-aged, high income, high spending')
print('Segment 2: Seniors, moderate income, low spending')

## Practice Exercises

Test your understanding of K-Means clustering with these exercises:

### Exercise 1
Generate synthetic data using `make_blobs` with 5 centers, a cluster standard deviation of 0.8, and 400 samples. Run K-Means with K from 2 to 10 and determine the optimal K using both the elbow method and the silhouette score. Do both methods agree?

### Exercise 2
Create a dataset with two features where one feature has values in [0, 1] and the other in [0, 1000]. Cluster the data with and without `StandardScaler`. Visualize both results side by side. Which features dominate in the unscaled clustering?

### Exercise 3
Use the `digits` dataset from `sklearn.datasets.load_digits`. Reduce its dimensionality to 2D using PCA, then cluster the digits with K-Means (K=10). Compare the clustering labels with the true digit labels using the Adjusted Rand Index and a confusion matrix. Which digits are most confusable?

### Exercise 4
Implement the **K-Means++ initialization** logic yourself (using the probability-based selection described in Section 9) and plug it into the `KMeansScratch` class. Test it on the blobs dataset and show that it produces lower inertia than random initialization on average.

### Exercise 5
Using the customer segmentation data from Section 13, add 10 extreme outliers (e.g., income of $1,000,000+). Run K-Means and K-Medoids. Which algorithm is less affected by the outliers? Compare the cluster centroids/medoids before and after adding outliers.